# Yodafone: Store Network Optimisation

**Important:** Set your student ID below and run the data generation cell before proceeding.

---

## Objective
Maximise profit by selecting which retail stores to reopen, accounting for:
- **ML forecasting** of zone demand (some zones have unknown demand)
- **Demand allocation** (overlapping stores split demand using your chosen rule)
- **Operational costs** (store opening costs reduce profit)
- **Network effects** (opening all stores may be suboptimal due to demand cannibalisation)

## Key Insight
Each store is **individually profitable** when opened alone. However, when multiple stores compete for the same zone's demand:
- Closer stores often capture more demand.
- More stores = wasted capacity (same total demand split among more outlets)
- Fewer stores = same demand with lower costs
- **Optimisation task**: Find the best subset of stores to maximise NETWORK profit

## Workflow
1. Generate personalised data
2. Load and explore data
3. Forecast missing zone demand using ML
4. Formulate and solve optimisation model
5. Analyse results and perform sensitivity analysis

---

## Assessment Overview

This coursework is worth **25% of your module grade** and is divided into two components:

### Notebook (80% of coursework grade)

Your Jupyter notebook will be marked on:

| Criterion | Weight | What's Assessed |
|-----------|--------|-----------------|
| **Demand Modelling & Forecasting** | 20% | Feature selection, model choice, validation, metrics interpretation, limitations |
| **Decision Modelling & Optimisation** | 20% | Formulation clarity, method justification, result validation, economic coherence |
| **Integration, Scenarios & Insight** | 20% | Forecast integration, scenario analysis, trade-off reasoning, business insight |
| **Understanding & Justification** | 15% | Clear explanation of modelling choices and results |
| **Technical Quality & Reproducibility** | 5% | Notebook runs end-to-end, clean structure, labelled outputs |

### Slides & Video (20% of coursework grade)

**Executive Slide Deck (5–6 slides, PDF)**
- Slide 1: Problem & opportunity
- Slide 2: Your recommendation (which stores to open)
- Slide 3: Value created (profit improvement vs. baseline)
- Slide 4: Trade-offs & scenarios
- Slide 5: Risks & next steps
- (Optional) Slide 6: Appendix / Q&A

**Video Presentation (10 ± 1 minute, narrated)**
- Walkthrough of slides and key decisions
- Cover: problem, recommendation, value, scenarios, risks, next steps
- Over/under-time submissions may be penalised

### Submission Details

- **Notebook**: Submit via GitHub Classroom
- **Slides (PDF)**: Submit via Blackboard
- **Video (MP4/similar)**: Submit via Blackboard
- **Deadline**: 8th May 2026, 12:00 (midday)

---

## 1. Set Student ID and Generate Data

**IMPORTANT:** Replace `XXXXXXXX` with your actual student number, then run the next cell to generate your personalised dataset.

In [ ]:
# ============================================================================
# SET YOUR STUDENT ID HERE
# ============================================================================
STUDENT_ID = "XXXXXXXX"  # Replace with your student number
# ============================================================================

# Generate your personalised dataset into the data/ folder.
# (Works in VS Code, Anaconda Jupyter, Google Colab and the course website.)
from pathlib import Path
import numpy, pandas  # load the packages generate_data uses (needed in the browser)
import generate_data

print(f"Generating data for student ID: {STUDENT_ID}")
generate_data.generate(
    student_id=str(STUDENT_ID),
    base_dir=Path("base_data"),
    out_dir=Path("data"),
    perturb=generate_data.DEFAULT_PERTURB,
    allow_distance_jitter=True,
)
print("Data generation successful!")

## 2. Import Libraries

In [ ]:
# Run this cell before the others.
# On the course website (JupyterLite) it installs this notebook's packages into the browser tab
# (about a minute the first time) and adapts them to the browser.
# On your own computer it does nothing: use the conda environment from environment.yml
# (see README.md).
# In Google Colab, run  !pip install pyomo highspy  instead.
import sys
if sys.platform == "emscripten":  # only in the browser
    import micropip
    await micropip.install(["pyomo==6.10.1", "highspy"])
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True
    # 3) Pyodide has no multiprocessing: stop Pyomo's import hook for it from breaking
    #    packages that try it and fall back (scikit-learn / joblib)
    import pyomo.common.dependencies as _deps
    for _ind in _deps.DeferredImportCallbackFinder._callbacks.get("multiprocessing", []):
        _ind._callback = None
print("Ready.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

## 3. Load Data

Your personalised dataset includes:
- `zones.csv`: 300 zones with socioeconomic features and partial demand data
- `stores.csv`: 80 stores with capacity and costs
- `zone_store_dist.csv`: Distance matrix (24,000 pairs)
- `margins.json`: Economic parameters

In [ ]:
# Load datasets
zones = pd.read_csv('data/zones.csv')
stores = pd.read_csv('data/stores.csv')
distances = pd.read_csv('data/zone_store_dist.csv')

with open('data/margins.json') as f:
    margins = json.load(f)
    margin_per_unit = margins['margin_per_unit']
    service_cost_per_km = margins['service_cost_per_km']

print(f"Zones: {len(zones)} rows")
print(f"  - With demand data: {zones['historical_sales'].notna().sum()}")
print(f"  - Missing demand: {zones['historical_sales'].isna().sum()}")
print(f"\nStores: {len(stores)} rows")
print(f"Distances: {len(distances)} rows")
print(f"\nEconomic parameters:")
print(f"  Margin per unit: £{margin_per_unit:.2f}")
print(f"  Service cost per km: £{service_cost_per_km:.3f}")

In [ ]:
# Basic statistics
print("Zone demand statistics (where available):")
print(zones[zones['historical_sales'].notna()]['historical_sales'].describe())

print("\n" + "="*60)
print("Store economics:")
print(stores[['capacity', 'fixed_opening_cost', 'service_radius_km']].describe())

In [ ]:
# Visualisation: Demand distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of demand (zones with data)
zones_with_demand = zones[zones['historical_sales'].notna()]
axes[0].hist(zones_with_demand['historical_sales'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Annual Demand (units)')
axes[0].set_ylabel('Number of Zones')
axes[0].set_title('Zone Demand Distribution')
axes[0].grid(True, alpha=0.3)

# Store opening costs
axes[1].hist(stores['fixed_opening_cost'], bins=20, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Opening Cost (£)')
axes[1].set_ylabel('Number of Stores')
axes[1].set_title('Store Opening Cost Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Geographic visualisation: zones and stores
plt.figure(figsize=(12, 8))

# Plot zones (colour by demand where available)
zones_with_demand = zones[zones['historical_sales'].notna()]
zones_missing = zones[zones['historical_sales'].isna()]

plt.scatter(zones_missing['lon'], zones_missing['lat'], 
           c='lightgray', s=30, alpha=0.5, label='Zones (missing demand)')
plt.scatter(zones_with_demand['lon'], zones_with_demand['lat'], 
           c=zones_with_demand['historical_sales'], cmap='YlOrRd', 
           s=50, alpha=0.6, label='Zones (known demand)')
plt.colorbar(label='Demand (units/year)')

# Plot stores
plt.scatter(stores['lon'], stores['lat'], 
           c='blue', marker='^', s=100, 
           edgecolors='black', linewidths=1, label='Stores', alpha=0.7)

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Geographic Distribution: Zones and Stores')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Milestone 1: Demand Forecasting

**Goal:** Predict annual demand for zones where `historical_sales` is missing.

**What you need to figure out:**
- Which features are most predictive of demand?
- What ML model is appropriate for this regression task?
- How will you validate your predictions are reasonable?
- Are there outliers or data quality issues to address?

**Deliverable:** A complete `zones` dataset with no missing demand values, plus evidence your forecasts are reliable.

---

*Add cells below to complete this milestone*

---
## 6. Milestone 2: Economic Baselines

**Goal:** Understand the profitability structure before building your optimisation model.

**Questions to answer:**
- What happens if you open ALL stores? (Calculate total profit)
- What happens if you open just ONE store? (Which is most profitable alone?)
- Why is opening all stores suboptimal?
- How does your demand allocation method create cannibalisation?

**Important:** Use the **same demand allocation method** here and in Milestone 3, so comparisons are fair.

**Deliverable:** Clear quantitative answers showing why optimisation is needed. Show your calculations.

---

*Add cells below to complete this milestone*

---
## 7. Milestone 3: Optimisation Model

**Goal:** Decide which stores to open to maximise network profit.

**Key decisions you must make:**
- How will you formulate this as an optimisation problem?
  - What are your decision variables?
  - What is your objective function?
  - What constraints are necessary?
- Will you use a heuristic, ILP, or MILP approach?
- How will you model demand allocation between competing stores?

**Demand allocation requirement:**
- You must define and justify a demand allocation method.
- Your method should use at least one operational driver (e.g., distance, capacity, service radius).
- You must use the same method consistently for baselines and optimisation comparisons.

**Examples (illustrative, not mandatory):**
- Gravity-style probabilistic split
- Nearest-open-store assignment
- Capacity-weighted allocation rule

**Hints:**
- Start simple, get a working model, then refine assumptions.
- Clearly state assumptions so your results are reproducible.

**Deliverable:** A working optimisation model that selects stores and reports total profit.

---

### Helpful: Simple Pyomo Example

Here's a minimal working example:

In [ ]:
import pyomo.environ as pyo

# Create model
model = pyo.ConcreteModel()

# Decision variables
model.x = pyo.Var([0, 1], domain=pyo.Binary)  # x[0], x[1] are binary

# Objective: maximise profit
model.profit = pyo.Objective(
    expr=10*model.x[0] + 15*model.x[1], 
    sense=pyo.maximize
)

# Constraint: budget limit
model.budget = pyo.Constraint(expr=5*model.x[0] + 8*model.x[1] <= 10)

# Solve
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)    # set tee=True to see solver output

print("Diplaying model information:")
model.display()

print("Displaying solution:")
model.pprint()


**Key concepts:**
- `pyo.Var()`: Decision variables (Binary, NonNegativeReals, Integers, etc.)
- `pyo.Objective()`: What to maximise/minimise
- `pyo.Constraint()`: Rules variables must satisfy
- `pyo.SolverFactory()`: Solver to use (HiGHS is recommended for ILP/MILP)

---

*Add cells below to complete this milestone*

## 8. Milestone 4: Solution Analysis

**Goal:** Interpret and validate your solution.

**What to investigate:**
- How many stores should open? Which ones?
- Where are the open stores geographically clustered?
- How much profit improvement vs. the "open all" baseline?
- Are there zones with poor coverage? Is this acceptable?
- Does your solution make business sense?

**Deliverable:** Clear explanation of your recommended store network, with supporting evidence (tables, maps, metrics).

---

*Add cells below to complete this milestone*

## 9. Milestone 5: Scenario and Sensitivity Analysis

**Goal:** Test how sensitive your solution is to uncertainties.

**Scenarios to explore (pick at least 2):**
- What if your demand forecasts are off by ±10-20%?
- What if opening costs increase due to inflation?
- What if margins shrink due to competition?
- What if you need to open at least X stores for brand presence?

**Deliverable:** Analysis showing whether your recommended stores remain optimal under different scenarios, or if the solution changes significantly.

---

*Add cells below to complete this milestone*

## 10. Modelling Reflection

### Reflection on Assumptions, Limitations, and Improvements (150–250 words)

Please refer to the assessment brief for guidance.

---

**[Replace this text with your reflection (150–250 words)]**

## 11. Executive Summary and Recommendations

### Summary and Recommendations (200–300 words)

Please refer to the assessment brief for guidance.

---

**[Replace this text with your executive summary (200–300 words)]**

---

## Checklist Before Submission

**Milestone Structure & Content**
- [ ] Milestone 1: Feature selection, model choice, validation approach, metrics, and limitations documented
- [ ] Milestone 2: All-stores baseline, single-store baseline, and cannibalisation evidence provided
- [ ] Milestone 3: Decision variables, constraints, objective function, solver, and assumptions stated
- [ ] Milestone 4: Store selection explained, profit vs. baseline compared, capacity and coverage analysed
- [ ] Milestone 5: At least 2 scenarios run and compared quantitatively
- [ ] Modelling Reflection: ~150–250 words covering assumption, limitation, improvement
- [ ] Summary and Recommendations: ~200–300 words covering network, value, trade-offs, risks

**Technical and Reproducibility**
- [ ] Notebook runs from top to bottom (Kernel → Restart & Run All)
- [ ] Student ID is set correctly at the top
- [ ] All cells have output (no unexecuted cells)
- [ ] Results are interpreted (not just numbers)
- [ ] Visualisations are clear and labeled
- [ ] Code is commented where needed
- [ ] Demand allocation method is explicitly defined and justified
- [ ] The same demand allocation method is used in baseline and optimisation sections
- [ ] All key assumptions documented and reproducible
- [ ] File is named exactly `notebook.ipynb`